In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
import statsmodels.api as sm
import numpy.linalg as npl


In [2]:
df = pd.read_csv('../output/data/final_dataset.csv')
df.head()

#total market by country
df['total_market'] = df.groupby('country')['new_est'].transform('sum')
df['s_j'] = df['new_est'] / df['total_market']

#compute market share of outside good
outside_by_country = df[df['nest'].str.upper() == 'OUTSIDE'].groupby('country')['new_est'].sum()
df['Q_out'] = df['country'].map(outside_by_country)
df['s_0'] = df['Q_out'] / df['total_market']

epsilon = 1e-10  # avoid log(0)
df['y'] = np.log(df['s_j'].clip(lower=epsilon)) - np.log(df['s_0'].clip(lower=epsilon))

df_us = df.loc[df['country'] == 'United States'].copy()

need = ["s_j", "price"]
df_us = df_us.replace([np.inf, -np.inf], np.nan).dropna(subset=need).copy()

alpha_hat = float(-0.4778)
firms = df_us['developer'].astype('category').cat.codes.to_numpy()
Omega_current = (firms[:, None] == firms[None, :]).astype(float)


In [3]:
s = df_us['s_j'].to_numpy().astype(float)
outer_ss = np.outer(s, s) 
p = df_us['price'].to_numpy().astype(float)
J = alpha_hat * (np.diag(s) - outer_ss)
n = len(df_us)

def calculate_markups(Omega, ownership): #define it to use it later in point 2.2
    #Markups from FOCs: 
    G = -(Omega * J)  # Hadamard product
    try:
        markup =  npl.solve(G, s)
    except npl.LinAlgError:
        # Slight ridge in case of near-singularity (e.g., |alpha| tiny) \\ io sta cosa toglierei
        lam = 1e-8
        markup =   npl.solve(G + lam*np.eye(n), s)

    mc = p - markup
    pct = np.where(np.isfinite(markup / p), markup / p, np.nan)
    #Save marginal costs and markups
    df_summary = pd.DataFrame(
        {
            "ownership": ownership,
            "markup": markup,
            "mc": mc,
            "pct_markup": pct,
            "developer": df_us["developer"].to_numpy(),
            "app": df_us["app"].to_numpy(),
        },
        index=df_us.index,   # ensure 1:1 join on index
    )
    return df_summary

In [4]:
y = df_us['y']
a = y - alpha_hat * df_us['price']

df_us['mc'] = calculate_markups(Omega_current, "current")['mc'].to_numpy()

In [5]:
df_us

,rank,app,developer,country,app_id,price,ndownloads,lowerbound,upperbound,averagescore,...,lowest_rank,multiplier,new_est,nest,total_market,s_j,Q_out,s_0,y,mc
3,397,EasyMSR,DEFTUN TECH,United States,com.gbtf.msrx6pro,19.99,"10,000+",10000,50000,3.5,...,500,81.466393,18391.0390,OUTSIDE,3.243590e+08,0.000057,3.804279e+07,0.117286,-7.634603,17.896955
10,366,"ArtRage: Draw, Paint, Create",Ambient Design Ltd.,United States,com.ambientdesign.artrage.playstore,4.99,"50,000+",50000,100000,4.1,...,499,106.157110,64118.8950,OUTSIDE,3.243590e+08,0.000198,3.804279e+07,0.117286,-6.385728,2.896660
39,429,AlfaOBD,AlfaOBD Soft,United States,com.AlfaOBD.AlfaOBD,49.00,"5,000+",5000,10000,4.2,...,477,11.210762,5538.1167,OUTSIDE,3.243590e+08,0.000017,3.804279e+07,0.117286,-8.834812,46.907038
44,309,inCarDoc Pro | ELM327 OBD2 Scanner Bluetooth/WiFi,inCarDoc,United States,com.pnn.obdcardoctor_full,2.99,"100,000+",100000,500000,4.5,...,498,808.080810,252727.2700,OUTSIDE,3.243590e+08,0.000779,3.804279e+07,0.117286,-5.014156,0.895442
51,414,Car Launcher Pro,apps lab studio,United States,com.autolauncher.motorcar,2.99,"50,000+",50000,100000,4.3,...,499,106.157110,59023.3550,OUTSIDE,3.243590e+08,0.000182,3.804279e+07,0.117286,-6.468534,0.896693
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5620,226,Hurricane Tracker,"EZ Apps, Inc",United States,com.steveparker.hurricaneTrackerAndroid,3.99,"10,000+",10000,50000,4.6,...,500,81.466393,32321.7910,OUTSIDE,3.243590e+08,0.000100,3.804279e+07,0.117286,-7.070725,1.896866
5621,155,RadarScope,DTN,United States,com.basevelocity.radarscope,9.99,"100,000+",100000,500000,4.1,...,498,808.080810,377171.7200,OUTSIDE,3.243590e+08,0.001163,3.804279e+07,0.117286,-4.613766,7.894638
5625,143,MyRadar Weather Radar Ad Free,ACME AtronOmatic LLC,United States,com.acmeaom.android.myradarpro,2.99,"50,000+",50000,100000,4.2,...,499,106.157110,87791.9300,OUTSIDE,3.243590e+08,0.000271,3.804279e+07,0.117286,-6.071497,0.896507
5640,449,NOAA Weather Unofficial (Pro),Granite Apps,United States,com.nstudio.weatherhere,1.99,"50,000+",50000,100000,4.6,...,499,106.157110,55307.8550,OUTSIDE,3.243590e+08,0.000171,3.804279e+07,0.117286,-6.533552,-0.103283


In [6]:

def shares_logit(p_vec: np.ndarray) -> np.ndarray:
    util = a + alpha_hat * p_vec
    expu = np.exp(util)
    denom = 1.0 + np.sum(expu)
    return expu / denom #return the shares

def compute_jacobian(s: np.ndarray) -> np.ndarray:
    outer_ss = np.outer(s, s)
    J = alpha_hat * (np.diag(s) - outer_ss)  # shape (n,n)
    return J

def solve_nevo_eq5(mc_vec, Omega, p_init=None, tol:float=1e-3, max_iter:int=1000, damping=0.6):
    """
    Solve Nevo (2000) equation (5):
        s(p) + (Ω ⊙ J(p)) (p - mc) = 0
    for equilibrium prices p.
    
    Args:
        mc_vec:   Marginal costs for each product.
        Omega:    Ownership matrix, where Ω[j,k] = 1 if products j and k are jointly owned.
        p_init:   Initial price vector. If None, it will use df_us['price'].
        tol:      Convergence tolerance for FOC residuals.
        max_iter: Maximum number of iterations.
        damping:  Relaxation factor (0.4–0.8 recommended).
    """
    # if no initial guess provided, start from your observed prices in df_us
    p = df_us["price"].to_numpy(float).copy() if p_init is None else np.array(p_init, float).copy()
    
    for i in range(max_iter):
        # Compute current market shares under current prices
        s_now = shares_logit(p)
        
        # Compute Jacobian ds/dp under simple logit
        J_now = compute_jacobian(s_now)

        # Step 3: Hadamard product
        G = -(Omega * J_now)

        # Step 4: fixed-point mapping -> solve G * step = s
        step = npl.solve(G, s_now)
        p_new = mc_vec + step

        # Step 5: apply damping (helps convergence)
        p = (1 - damping) * p + damping * p_new

        # Step 6: compute FOC residuals F(p) = s + (Ω⊙J)(p - mc)
        res = s_now + (Omega * J_now) @ (p - mc_vec)
        # Step 7: check convergence
        if np.max(np.abs(res)) < tol:
            print(f"Converged in {i+1} iterations, max residual = {np.max(np.abs(res)):.2e}")
            return p, True

    # If not converged
    print("Did not converge; try increasing max_iter or adjusting damping.")
    return p, False

In [7]:
unique_developers = df_us['developer'].unique()
targets = [f for f in unique_developers if f != 'Mojang']

rows = []
for j, t in enumerate(targets):
    print('target number', j,'/',len(targets), ': ', t)
    # Make a copy of the us_market DataFrame for the current developer
    developer_data = df_us.copy()
    #update name of t to mojang
    developer_data.loc[developer_data['developer'] == t, 'developer'] = 'Mojang'

    #compute Omega under this ownership structure
    firms = developer_data['developer'].astype('category').cat.codes.to_numpy()
    Omega_post = (firms[:, None] == firms[None, :]).astype(float)
    #solve for new prices
    p_eq, converged = solve_nevo_eq5(
        mc_vec=df_us['mc'].to_numpy(float),
        Omega=Omega_post,
        p_init=df_us['price'].to_numpy(float),
        tol=1e-5,
        max_iter=1000,
        damping=0.5
    )
    #store results and profits
    mojang_mask = developer_data['developer'].eq('Mojang').to_numpy()
    rows.append({"target": t, "converged": bool(converged),
             "mojang_prices": p_eq[mojang_mask].tolist(),
             "mojang_profit": float(np.sum((p_eq - df_us['mc'].to_numpy(float)) * shares_logit(p_eq) * mojang_mask))}) #to check if shares are correct

target number 0 / 372 :  DEFTUN TECH
Converged in 7 iterations, max residual = 8.12e-06
target number 1 / 372 :  Ambient Design Ltd.
Converged in 7 iterations, max residual = 8.17e-06
target number 2 / 372 :  AlfaOBD Soft
Converged in 7 iterations, max residual = 8.11e-06
target number 3 / 372 :  inCarDoc
Converged in 7 iterations, max residual = 8.36e-06
target number 4 / 372 :  apps lab studio
Converged in 7 iterations, max residual = 8.16e-06
target number 5 / 372 :  Moon+
Converged in 7 iterations, max residual = 8.82e-06
target number 6 / 372 :  Workman Consulting LLC
Converged in 7 iterations, max residual = 8.17e-06
target number 7 / 372 :  HarperCollins Christian Publishing
Converged in 7 iterations, max residual = 8.11e-06
target number 8 / 372 :  Wolfram Group
Converged in 8 iterations, max residual = 6.68e-06
target number 9 / 372 :  Tecarta, Inc.
Converged in 7 iterations, max residual = 8.11e-06
target number 10 / 372 :  Merriam-Webster Inc.
Converged in 7 iterations, max 

In [8]:
df_results = pd.DataFrame(rows)
df_results.sort_values(by='mojang_profit', ascending=False)

,target,converged,mojang_prices,mojang_profit
63,Rockstar Games,True,"[5.320208946692192, 7.320208946692192, 7.32020...",0.436277
144,Fireproof Games,True,"[7.012047823504804, 2.323393585014011, 1.32339...",0.403326
81,SQUARE ENIX Ltd,True,"[1.3241434290941152, 7.009914942433094]",0.401261
202,ninja kiwi,True,"[7.009275072794089, 3.323664044362122, 5.32366...",0.400556
79,Bravestars Games,True,"[1.3251096455045626, 6.99452428287333]",0.385823
...,...,...,...,...
133,My Town Games Ltd,True,"[6.949710102905638, 3.3281448274140626]",0.341026
207,Open Lab Games,True,"[6.94970717879141, 6.328145104358288]",0.341023
270,Ahmed Bousrih,True,"[6.949706740175073, 2.3281451458998674]",0.341023
73,Blazes,True,"[1.828145282063839, 6.949705302489609]",0.341022
